# 02 Hypothesis Testing and Statistical Inference

This notebook uses the raw dataset, not the SMOTE-balanced data. It is meant for paper writing: t-tests, chi-square tests, correlation analysis, ANOVA, and logistic regression.

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency, pearsonr, f_oneway
import statsmodels.formula.api as smf

In [ ]:
from google.colab import files
print('Select dataset (CSV) to upload when prompted.')
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
target_col = 'depression_label'
print('Raw inference dataset shape:', df.shape)
print(df.head())

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.formula.api import logit

# Target column
TARGET = 'depression_label'
# Features used in the inference summary
numeric_features = [
    'age',
    'daily_social_media_hours',
    'sleep_hours',
    'screen_time_before_sleep',
    'academic_performance',
    'physical_activity',
    'stress_level',
    'anxiety_level',
    'addiction_level',
]
categorical_features = ['gender', 'platform_usage', 'social_interaction_level']

def cohens_d(group0, group1):
    n0, n1 = len(group0), len(group1)
    if n0 < 2 or n1 < 2:
        return np.nan
    sd0, sd1 = group0.std(ddof=1), group1.std(ddof=1)
    pooled_var = ((n0 - 1) * sd0**2 + (n1 - 1) * sd1**2) / (n0 + n1 - 2)
    if pooled_var <= 0 or np.isnan(pooled_var):
        return np.nan
    return (group1.mean() - group0.mean()) / np.sqrt(pooled_var)


def cramers_v(table):
    chi2, _, _, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    denom = min(r - 1, k - 1)
    if n == 0 or denom <= 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)


def eta_squared_from_f(f_stat, df_between, df_within):
    if np.isnan(f_stat) or df_between <= 0 or df_within <= 0:
        return np.nan
    return (f_stat * df_between) / (f_stat * df_between + df_within)


print('1) Welch t-tests for numeric features vs depression_label')
t_rows = []
for col in numeric_features:
    group0 = df.loc[df[TARGET] == 0, col].dropna()
    group1 = df.loc[df[TARGET] == 1, col].dropna()
    if len(group0) > 1 and len(group1) > 1:
        t_stat, p_val = stats.ttest_ind(group0, group1, equal_var=False)
        d_val = cohens_d(group0, group1)
        t_rows.append({
            'feature': col,
            't_stat': t_stat,
            'p_value': p_val,
            'cohens_d': d_val,
            'abs_cohens_d': abs(d_val) if not np.isnan(d_val) else np.nan,
        })
        print(f'{col}: t={t_stat:.3f}, p={p_val:.4g}, d={d_val:.3f}')


t_summary = pd.DataFrame(t_rows).sort_values('abs_cohens_d', ascending=False)
print('\nTop numeric effects by |Cohen\'s d|:')
print(t_summary[['feature', 'p_value', 'cohens_d', 'abs_cohens_d']].head().to_string(index=False))

print('\n2) Chi-square tests for categorical features vs depression_label')
chi_rows = []
for col in categorical_features:
    table = pd.crosstab(df[col], df[TARGET])
    chi2, p_val, dof, expected = stats.chi2_contingency(table)
    v_val = cramers_v(table)
    chi_rows.append({
        'feature': col,
        'chi2': chi2,
        'dof': dof,
        'p_value': p_val,
        'cramers_v': v_val,
    })
    print(f'{col}: chi2={chi2:.3f}, dof={dof}, p={p_val:.4g}, V={v_val:.3f}')

chi_summary = pd.DataFrame(chi_rows).sort_values('p_value')

print('\n3) Point-biserial correlation for numeric features vs depression_label')
cor_rows = []
for col in numeric_features:
    r, p_val = stats.pearsonr(df[col], df[TARGET])
    cor_rows.append({'feature': col, 'r': r, 'p_value': p_val})
    print(f'{col}: r={r:.3f}, p={p_val:.4g}')

cor_summary = pd.DataFrame(cor_rows).sort_values('p_value')

print('\n4) ANOVA example: sleep_hours across platform_usage groups')
# Chosen numeric outcome for the ANOVA example: sleep_hours
anova_outcome = 'sleep_hours'
anova_group = 'platform_usage'
groups = [grp[anova_outcome].dropna() for _, grp in df.groupby(anova_group)]
if len(groups) >= 2 and all(len(g) > 1 for g in groups):
    f_stat, p_val = stats.f_oneway(*groups)
    df_between = len(groups) - 1
    df_within = sum(len(g) for g in groups) - len(groups)
    eta_sq = eta_squared_from_f(f_stat, df_between, df_within)
    print(f'{anova_outcome} by {anova_group}: F={f_stat:.3f}, p={p_val:.4g}, eta_sq={eta_sq:.3f}')
else:
    print(f'ANOVA skipped for {anova_outcome} by {anova_group}: insufficient group sizes.')

print('\n5) Logistic regression example')
formula = (
    'depression_label ~ age + daily_social_media_hours + sleep_hours + '
    'screen_time_before_sleep + academic_performance + physical_activity + '
    'stress_level + anxiety_level + addiction_level + '
    'C(gender) + C(platform_usage) + C(social_interaction_level)'
)
try:
    model = logit(formula, data=df.dropna()).fit(disp=False)
    # Compact model summary (safer nobs retrieval)
    if hasattr(model, 'nobs'):
        nobs = int(model.nobs)
    else:
        # fall back to model.model.endog length (works for ModelResults/Result objects)
        try:
            nobs = int(getattr(model, 'model').endog.shape[0])
        except Exception:
            nobs = float('nan')
    llf = getattr(model, 'llf', float('nan'))
    prs = getattr(model, 'prsquared', float('nan'))
    converged = bool(getattr(model, 'converged', True))
    print(f'Logistic regression: Nobs={nobs}, LL={llf:.3f}, Pseudo R2={prs:.3f}, converged={converged}')

    coef_table = pd.DataFrame({
        'coef': model.params,
        'std_err': model.bse,
        'z': model.tvalues,
        'p_value': model.pvalues
    })
    print('\nTop coefficients by p-value:')
    print(coef_table.sort_values('p_value').head(10).to_string())

    odds_ratios = pd.DataFrame({
        'coef': model.params,
        'odds_ratio': np.exp(model.params),
        'ci_lower': np.exp(model.conf_int()[0]),
        'ci_upper': np.exp(model.conf_int()[1]),
        'p_value': model.pvalues,
    })
    print('\nLogistic regression odds ratios:')
    print(odds_ratios.sort_values('p_value').head(10).to_string())
except Exception as e:
    print('Logit failed:', e)
